# Student 4 

**UTS Deep Learning Assessment 3 (Group 1) — VizWiz Image Captioning**

This notebook contains my individual contributions:

- **Model 1**: ResNet-101 (frozen) + OCR text features + single-layer LSTM decoder + beam search decoding.
- **Model 2**: To be added after the group discussion of Phase 2 results.

The shared data preparation notebook (Phase 1) produces the cleaned dataset, vocabulary, and splits that this notebook consumes.

**Prerequisites**

1. Pull the latest shared Phase 1 data preparation notebook from the group's branch.
2. Run it to produce `data/processed/vizwiz_cleaned.json` and `data/processed/vocab.json`.
3. Install OCR + vision dependencies: `poetry add torchvision easyocr`. EasyOCR pulls a few pretrained models (~150 MB) on first import.
4. Select the **Python (dl-at3-group1)** kernel and **Kernel → Restart** before running.

**Runtime**: ~75-95 min on Apple Silicon MPS:
- ~15 min one-time OCR pre-extraction over 7,324 images (cached to `data/processed/ocr_text.json` — subsequent runs skip this)
- ~30-40 min training (15 epochs)
- ~20-30 min BLEU evaluation with beam search on train sample + val + test


---
# **Table of Contents**
---

> [1. Setup](#1-setup)  
> [2. Data Pipeline](#2-data)  
> [3. OCR Pre-Extraction](#3-ocr)  
> [4. Model 1 — Architecture](#4-architecture)  
> [5. Model 1 — Training](#5-training)  
> [6. Model 1 — Sample Generations](#6-samples)  
> [7. Model 1 — BLEU Evaluation](#7-bleu)  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.1 Train vs Val vs Test — Overfit Diagnostic  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.2 Per-image BLEU distribution  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.3 Best vs worst predictions  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.4 Caption length comparison  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.5 Vocabulary usage  
> &nbsp;&nbsp;&nbsp;&nbsp; 7.6 Summary table  
> [8. Model 1 — Summary & Limitations](#8-summary)  
> [9. Group Discussion Notes](#9-discussion)  
> [10. Model 2 — Refined Architecture (TBD)](#10-model2)  


---
# **1. Setup**
---


**Methodology**

Standard PyTorch + torchvision imports plus NLTK for BLEU evaluation and EasyOCR for text extraction. All hyperparameters are declared up-front for one-place tuning. Hardware acceleration is detected automatically (MPS / CUDA / CPU).

`NUM_WORKERS = 0` is enforced because notebook-defined classes cannot be pickled to spawn-mode worker processes on macOS Python 3.13+.

**Hyperparameter choices for this architecture (OCR + LSTM + Beam):**
- `LR = 1e-3` — standard LR for RNN decoders (Transformer architectures need lower, but LSTMs handle this fine).
- `EPOCHS = 15` — gives LSTM time to converge with the additional OCR feature stream.
- `EMBED_DIM = 256`, `HIDDEN_DIM = 512` — canonical Show-and-Tell decoder configuration.
- `OCR_HIDDEN_DIM = 256` — separate LSTM that encodes the extracted OCR text into a fixed vector.
- `OCR_MAX_LEN = 20` — most VizWiz OCR strings are short (medication labels, product names); truncation beyond this is fine.
- `BEAM_SIZE = 5` — standard captioning beam width; trades a few seconds of inference time for typically +1-2 BLEU-4 over greedy.
- `LENGTH_PENALTY_ALPHA = 0.7` — Wu et al. 2016 length normalisation: `score / len^alpha`. Prevents beam search from systematically preferring shorter outputs.


In [ ]:
# Imports necessary libraries
import json, random, time, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torchvision import transforms, models
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

# Paths
IMG_DIR      = Path('../data/raw/val')
CLEANED_JSON = Path('../data/processed/vizwiz_cleaned.json')
VOCAB_JSON   = Path('../data/processed/vocab.json')
OCR_CACHE    = Path('../data/processed/ocr_text.json')  # pre-extracted OCR cached here



In [ ]:
# Hyperparameters
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 0
MIN_COUNT    = 5
MAX_SEQ_LEN  = 22

# LSTM decoder + OCR encoder
EMBED_DIM       = 256
HIDDEN_DIM      = 512
FEATURE_DIM     = 2048    # ResNet-101 final feature size (global avg pool)
OCR_EMBED_DIM   = 128
OCR_HIDDEN_DIM  = 256
OCR_MAX_LEN     = 20

# Decoding (beam search)
BEAM_SIZE             = 5
LENGTH_PENALTY_ALPHA  = 0.7

# Training parameters
EPOCHS         = 15
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
GRAD_CLIP      = 5.0      
DROPOUT        = 0.1

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
# **2. Data Pipeline**
---


**Methodology**

Data infrastructure rebuilt inline so the notebook is self-contained. Classes mirror the shared Phase 1 notebook but extend the Dataset/Collate to also carry OCR token sequences alongside each image and caption.

- `Vocabulary`: word-level mapping with `<pad>` / `<bos>` / `<eos>` / `<unk>`, built from the training split only.
- `VizWizCaptionsWithOCR`: PyTorch `Dataset` returning `(image, caption_ids, caption_length, ocr_ids, ocr_length)`.
- `CollateOCR`: pads both caption and OCR sequences to longest-in-batch independently.


In [ ]:
# ocabulary Class & Tokenizer
def tokenize(s): return s.split()

class Vocabulary:
    PAD, BOS, EOS, UNK = '<pad>', '<bos>', '<eos>', '<unk>'
    SPECIALS = [PAD, BOS, EOS, UNK]

    def __init__(self, word2idx, min_count):
        self.word2idx = word2idx
        self.idx2word = {i: w for w, i in word2idx.items()}
        self.min_count = min_count

    @classmethod
    def build(cls, tokenized_captions, min_count=5):
        cnt = Counter(t for toks in tokenized_captions for t in toks)
        kept = sorted([w for w, c in cnt.items() if c >= min_count])
        w2i = {tok: i for i, tok in enumerate(cls.SPECIALS)}
        for w in kept: w2i[w] = len(w2i)
        return cls(w2i, min_count)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            blob = json.load(f)
        return cls(blob['word2idx'], blob['min_count'])

    def numericalize(self, tokens, add_bos_eos=True, max_len=None):
        ids = [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens]
        if add_bos_eos: ids = [self.word2idx[self.BOS]] + ids + [self.word2idx[self.EOS]]
        if max_len is not None: ids = ids[:max_len]
        return ids

    def denumericalize(self, ids, strip_special=True):
        toks = [self.idx2word[i] for i in ids if i in self.idx2word]
        if strip_special: toks = [t for t in toks if t not in self.SPECIALS]
        return ' '.join(toks)

    def __len__(self): return len(self.word2idx)
    @property
    def pad_idx(self): return self.word2idx[self.PAD]
    @property
    def bos_idx(self): return self.word2idx[self.BOS]
    @property
    def eos_idx(self): return self.word2idx[self.EOS]
    @property
    def unk_idx(self): return self.word2idx[self.UNK]

In [ ]:
# Load Cleaned Dataset & Vocabulary
if not CLEANED_JSON.exists():
    raise FileNotFoundError(f'{CLEANED_JSON} not found. Run the shared Phase 1 data prep notebook first.')

with open(CLEANED_JSON) as f:
    records = json.load(f)
for r in records:
    if 'cleaned_captions' not in r and 'captions' in r:
        r['cleaned_captions'] = r['captions']

by_split = {'train': [], 'val': [], 'test': []}
for r in records: by_split[r['split']].append(r)

if VOCAB_JSON.exists():
    vocab = Vocabulary.load(VOCAB_JSON)
    print(f'Loaded vocab from {VOCAB_JSON}')
else:
    train_tokens = [tokenize(c) for r in by_split['train'] for c in r['cleaned_captions']]
    vocab = Vocabulary.build(train_tokens, min_count=MIN_COUNT)
    print(f'Built vocab from training split (vocab.json missing at {VOCAB_JSON})')
VOCAB_SIZE = len(vocab)

print(f'  Vocab size : {VOCAB_SIZE:,}')
for k, v in by_split.items():
    print(f'  {k:5s} : {len(v):>5} images')

---
# **3. OCR Pre-Extraction**
---


**Methodology**

VizWiz images frequently contain visible text (medication labels, packaging, screens, mail). A pure visual model can only *infer* the existence of text from pixel patterns; an OCR-augmented model can directly *read* that text and inject it into the caption generation process.

**Why pre-extract once and cache** instead of running OCR per training step:
- EasyOCR is slow (~0.1-0.5 s per image on CPU). Running OCR inside the training loop would dominate runtime and gain nothing — OCR output for a given image is fixed and doesn't benefit from re-running each epoch.
- Pre-extraction lets us treat OCR text as a *static feature* of each image, like the filename. We extract once over all ~7,300 images (~15 min one-time cost), save to `data/processed/ocr_text.json`, and reload from the cache every subsequent run.

**Cleaning**: extracted OCR text is lowercased and stripped of non-alphanumeric characters (except whitespace). Matches the caption-vocabulary normalisation so OCR tokens align with the same `word2idx` mapping. OCR words that aren't in the training vocabulary become `<unk>` — acceptable, because the *presence* of OCR signal still helps the LSTM, even if specific brand names are out-of-vocabulary.

**Empty OCR fallback**: images with no detected text get a single `<pad>` token (cannot be empty — the OCR encoder LSTM needs at least one input).


In [ ]:
# Extract OCR text (cached)
def clean_ocr(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

if OCR_CACHE.exists():
    with open(OCR_CACHE) as f:
        ocr_dict = json.load(f)
    print(f'Loaded OCR cache: {len(ocr_dict):,} images.')
else:
    print('Extracting OCR for all images. One-time cost: ~10-15 min on CPU.')
    print('(EasyOCR will download its pretrained models on first run.)')
    import easyocr
    reader = easyocr.Reader(['en'], gpu=False)   # CPU is fine; only run once

    ocr_dict = {}
    for r in tqdm(records, desc='OCR'):
        fn = r['filename']
        try:
            results = reader.readtext(str(IMG_DIR / fn), detail=0, paragraph=True)
            text = ' '.join(results) if results else ''
        except Exception:
            text = ''
        ocr_dict[fn] = clean_ocr(text)

    with open(OCR_CACHE, 'w') as f:
        json.dump(ocr_dict, f)
    print(f'Saved OCR cache to {OCR_CACHE}')

# Sanity check: how many images have non-empty OCR?
with_text = sum(1 for v in ocr_dict.values() if v)
print(f'\n  Images with non-empty OCR : {with_text:,} / {len(ocr_dict):,} ({with_text/len(ocr_dict)*100:.1f}%)')
lengths = [len(v.split()) for v in ocr_dict.values() if v]
if lengths:
    print(f'  OCR token count          : mean={np.mean(lengths):.1f}, median={np.median(lengths):.1f}, max={max(lengths)}')

# Sample some OCR outputs for sanity
print('\nSample OCR extractions:')
for r in random.sample([r for r in records if ocr_dict.get(r['filename'])], 5):
    print(f"  {r['filename']:35s} -> {ocr_dict[r['filename']][:80]}")

In [ ]:
# Dataset & Collate Classes (with OCR)
class VizWizCaptionsWithOCR(Dataset):
    """Returns image + caption ids + OCR ids per item."""
    def __init__(self, records, img_dir, vocab, transform, max_len, ocr_dict,
                 ocr_max_len=OCR_MAX_LEN, random_caption=True):
        self.records, self.img_dir = records, Path(img_dir)
        self.vocab, self.transform = vocab, transform
        self.max_len, self.random_caption = max_len, random_caption
        self.ocr_dict, self.ocr_max_len = ocr_dict, ocr_max_len

    def __len__(self): return len(self.records)

    def _encode_ocr(self, fn):
        """OCR text -> token ids using the caption vocab (OOV -> <unk>)."""
        text = self.ocr_dict.get(fn, '')
        tokens = text.split()[:self.ocr_max_len]
        ids = [self.vocab.word2idx.get(t, self.vocab.unk_idx) for t in tokens]
        if not ids:                       # empty OCR -> single pad (LSTM needs len >= 1)
            ids = [self.vocab.pad_idx]
        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self.transform(Image.open(self.img_dir / r['filename']).convert('RGB'))
        cap = random.choice(r['cleaned_captions']) if self.random_caption else r['cleaned_captions'][0]
        cap_ids = torch.tensor(self.vocab.numericalize(tokenize(cap), max_len=self.max_len), dtype=torch.long)
        ocr_ids = self._encode_ocr(r['filename'])
        return img, cap_ids, len(cap_ids), ocr_ids, len(ocr_ids)


class CollateOCR:
    """Pads captions and OCR sequences independently to longest-in-batch."""
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs, caps, cap_lens, ocrs, ocr_lens = zip(*batch)
        return (torch.stack(imgs),
                pad_sequence(caps, batch_first=True, padding_value=self.pad_idx),
                torch.tensor(cap_lens, dtype=torch.long),
                pad_sequence(ocrs, batch_first=True, padding_value=self.pad_idx),
                torch.tensor(ocr_lens, dtype=torch.long))

In [ ]:
# Image Transforms & DataLoaders
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = VizWizCaptionsWithOCR(by_split['train'], IMG_DIR, vocab, train_tf, MAX_SEQ_LEN, ocr_dict, random_caption=True)
val_ds   = VizWizCaptionsWithOCR(by_split['val'],   IMG_DIR, vocab, eval_tf,  MAX_SEQ_LEN, ocr_dict, random_caption=False)

collate = CollateOCR(vocab.pad_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, collate_fn=collate, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate, num_workers=4, pin_memory=True)

print(f'  Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

---
# **4. Model 1 — Architecture**
---


**Summary Insights**

Model 1 fuses **two information streams** into a single LSTM decoder:
1. **Visual features** from a frozen pretrained ResNet-101 (2,048-d global pool vector per image).
2. **OCR features** from a small LSTM that reads the OCR text extracted from each image (256-d final hidden state).

These two vectors are concatenated and projected to initialise the decoder LSTM's hidden + cell states. The decoder then generates captions one token at a time, with **beam search (k=5, length-penalty α=0.7)** at inference time for higher-quality output than greedy.

**Design Rationale**

| Component | Choice | Why |
|---|---|---|
| Image encoder | ResNet-101, fully frozen | Matches the group's CNN convention (students 1, 2, 3, 5 also use ResNet-101). Frozen because 5,500 training images is too few to fine-tune safely, and the OCR + decoder paths already add significant trainable capacity. |
| Visual features | Global pool (2,048-d), not spatial | An LSTM decoder doesn't naturally consume spatial features (that's a Transformer/attention-decoder pattern). Global features are the right granularity for an init-hidden-state conditioning. |
| OCR backbone | EasyOCR, English | Pure-Python (no external Tesseract install), uses small PyTorch detection + recognition models. Adequate for the medium-resolution VizWiz images. |
| OCR encoder | Single-layer LSTM (`hidden_dim=256`) | Encodes the variable-length OCR text into a fixed 256-d vector via `pack_padded_sequence` for proper length handling. Smaller than the caption decoder — OCR signal is auxiliary, not the main story. |
| OCR fusion | Concat (image, OCR) → project → LSTM init state | Early fusion at the decoder's initial hidden state. Simple, well-precedented for multimodal LSTM captioning. |
| Caption decoder | Single-layer LSTM (`embed_dim=256`, `hidden_dim=512`) | Classical Show-and-Tell decoder, drops in directly with the fused init state. |
| Loss | Cross-entropy with `ignore_index=pad_idx` | Pad positions skipped automatically. |
| Optimizer | AdamW, `lr=1e-3` | Standard LR for RNN decoders. |
| Gradient clipping | `max_norm=5.0` | LSTMs tolerate a looser clip than Transformers — 5.0 is the canonical value (Pascanu et al. 2013). |
| Decoding | **Beam search**, `k=5`, length penalty α=0.7 | Per the group's spec. Beam search typically gives +1-2 BLEU-4 over greedy by exploring multiple high-probability continuations rather than committing to argmax at every step. |

**Distinction from teammate #1 (CNN + LSTM + fine-tune + beam)**: both use ResNet-101 + LSTM + beam search, but #1 fine-tunes the encoder while this Model 1 adds OCR text as auxiliary input. The comparison: *does OCR text help captioning more than encoder fine-tuning?*


In [ ]:
# Image Encoder — Frozen ResNet-101 (Global Pool)
class ResNet101Encoder(nn.Module):
    """Frozen ResNet-101, returns (B, 2048) global pooled features."""
    def __init__(self):
        super().__init__()
        backbone = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        # Keep everything up to and including avgpool; drop the final fc.
        self.feat = nn.Sequential(*list(backbone.children())[:-1])
        for p in self.feat.parameters():
            p.requires_grad = False

    def train(self, mode=True):
        # Keep BN in eval mode so ImageNet running stats are used, not noisy batch stats.
        super().train(mode)
        self.feat.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            x = self.feat(x)              # (B, 2048, 1, 1)
        return x.squeeze(-1).squeeze(-1)  # (B, 2048)

In [ ]:
# OCR Encoder — Small LSTM over OCR Tokens
class OCREncoder(nn.Module):
    """Encodes variable-length OCR token sequence into a fixed-size vector."""
    def __init__(self, vocab_size, embed_dim=OCR_EMBED_DIM, hidden_dim=OCR_HIDDEN_DIM, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.hidden_dim = hidden_dim

    def forward(self, ocr_ids, ocr_lens):
        # ocr_ids: (B, T_ocr), ocr_lens: (B,)
        emb = self.embedding(ocr_ids)
        packed = pack_padded_sequence(emb, ocr_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return h_n.squeeze(0)             # (B, hidden_dim)

In [ ]:
#LSTM Decoder with Beam Search Inference
class LSTMDecoder(nn.Module):
    """Single-layer LSTM decoder; fused image + OCR features initialise the hidden state."""
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                 img_feature_dim=FEATURE_DIM, ocr_feature_dim=OCR_HIDDEN_DIM,
                 dropout=DROPOUT, pad_idx=0):
        super().__init__()
        combined_dim = img_feature_dim + ocr_feature_dim    # 2048 + 256 = 2304
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.feat_to_h = nn.Linear(combined_dim, hidden_dim)
        self.feat_to_c = nn.Linear(combined_dim, hidden_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout   = nn.Dropout(dropout)
        self.proj      = nn.Linear(hidden_dim, vocab_size)

    def init_hidden(self, img_feats, ocr_feats):
        combined = torch.cat([img_feats, ocr_feats], dim=-1)
        h0 = torch.tanh(self.feat_to_h(combined)).unsqueeze(0)   # (1, B, H)
        c0 = torch.tanh(self.feat_to_c(combined)).unsqueeze(0)
        return h0, c0

    def forward(self, img_feats, ocr_feats, captions):
        h, c = self.init_hidden(img_feats, ocr_feats)
        emb = self.embedding(captions)
        out, _ = self.lstm(emb, (h, c))
        return self.proj(self.dropout(out))                      # (B, T, V)

    @torch.no_grad()
    def generate_beam(self, img_feats, ocr_feats, vocab, beam_size=BEAM_SIZE,
                      max_len=MAX_SEQ_LEN, alpha=LENGTH_PENALTY_ALPHA):
        """Beam search decoding, one image at a time. Returns list[list[int]] across batch."""
        self.eval()
        B = img_feats.size(0)
        device = img_feats.device
        results = []

        for b in range(B):
            h_init, c_init = self.init_hidden(img_feats[b:b+1], ocr_feats[b:b+1])
            # Each beam: (token_list, cumulative_log_prob, h, c, finished)
            beams = [([vocab.bos_idx], 0.0, h_init, c_init, False)]

            for _ in range(max_len):
                candidates = []
                for tokens, score, h, c, done in beams:
                    if done:
                        candidates.append((tokens, score, h, c, True))
                        continue
                    last_tok = torch.tensor([[tokens[-1]]], device=device)
                    emb = self.embedding(last_tok)
                    out, (h_new, c_new) = self.lstm(emb, (h, c))
                    log_probs = F.log_softmax(self.proj(out[0, -1]), dim=-1)
                    top_lp, top_ids = log_probs.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_ids.tolist()):
                        new_tokens = tokens + [tid]
                        new_score = score + lp
                        new_done = (tid == vocab.eos_idx)
                        candidates.append((new_tokens, new_score, h_new, c_new, new_done))

                # Rank by length-normalised score: divide by len^alpha
                candidates.sort(key=lambda x: x[1] / max(len(x[0]), 1)**alpha, reverse=True)
                beams = candidates[:beam_size]
                if all(beam[4] for beam in beams):
                    break

            # Best beam (by the same length-normalised criterion)
            beams.sort(key=lambda x: x[1] / max(len(x[0]), 1)**alpha, reverse=True)
            best_tokens = beams[0][0]
            # Strip BOS and stop at first EOS
            seq = []
            for t in best_tokens[1:]:
                if t == vocab.eos_idx: break
                seq.append(t)
            results.append(seq)
        return results

In [ ]:
# Captioning Model (Image Encoder + OCR Encoder + LSTM Decoder)
class CaptioningModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, ocr_embed_dim, ocr_hidden_dim,
                 dropout, pad_idx):
        super().__init__()
        self.encoder     = ResNet101Encoder()
        self.ocr_encoder = OCREncoder(vocab_size, ocr_embed_dim, ocr_hidden_dim, pad_idx)
        self.decoder     = LSTMDecoder(vocab_size, embed_dim, hidden_dim,
                                       img_feature_dim=2048, ocr_feature_dim=ocr_hidden_dim,
                                       dropout=dropout, pad_idx=pad_idx)

    def forward(self, images, ocr_ids, ocr_lens, captions):
        img_feats = self.encoder(images)
        ocr_feats = self.ocr_encoder(ocr_ids, ocr_lens)
        return self.decoder(img_feats, ocr_feats, captions)


model = CaptioningModel(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    ocr_embed_dim=OCR_EMBED_DIM, ocr_hidden_dim=OCR_HIDDEN_DIM,
    dropout=DROPOUT, pad_idx=vocab.pad_idx,
).to(device)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total parameters     : {n_total:>11,}')
print(f'  Trainable parameters : {n_trainable:>11,}  (OCR encoder + decoder; image encoder frozen)')

---
# **5. Model 1 — Training**
---


**Methodology**

Standard autoregressive training with teacher forcing — but the batch now carries OCR sequences alongside images and captions.

- `inputs = captions[:, :-1]` (everything before final `<eos>`)
- `targets = captions[:, 1:]` (everything after initial `<bos>`)
- Cross-entropy loss with `ignore_index=pad_idx` skips padding positions automatically.
- Both the OCR encoder LSTM and the caption decoder LSTM train end-to-end via backprop through the fused init-hidden-state path.
- Image encoder (ResNet-101) stays frozen throughout — no gradients flow through it.

Per-epoch validation loss is reported alongside training loss to monitor for overfitting.


In [ ]:
# Train + Validate Loop
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY,
)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, total_tokens = 0.0, 0
    for imgs, caps, _cap_lens, ocrs, ocr_lens in tqdm(loader, desc='train' if train else 'val ', leave=False):
        imgs = imgs.to(device); caps = caps.to(device)
        ocrs = ocrs.to(device); ocr_lens = ocr_lens.to(device)
        inputs, targets = caps[:, :-1], caps[:, 1:]
        with torch.set_grad_enabled(train):
            logits = model(imgs, ocrs, ocr_lens, inputs)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        n_tok = (targets != vocab.pad_idx).sum().item()
        total_loss   += loss.item() * n_tok
        total_tokens += n_tok
    return total_loss / max(total_tokens, 1)

history = {'train': [], 'val': []}
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    history['train'].append(tr); history['val'].append(va)
    print(f'  Epoch {epoch:2d} | train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s')

In [ ]:
# Loss Curves**
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS+1), history['train'], '-o', label='train', color='steelblue')
ax.plot(range(1, EPOCHS+1), history['val'],   '-o', label='val',   color='coral')
ax.set_xlabel('epoch'); ax.set_ylabel('loss (per-token CE)')
ax.set_title('Model 1 — ResNet-101 + OCR + LSTM training curve', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

---
# **6. Model 1 — Sample Generations**
---


**Methodology**

Six random **test-set** images are sampled and their predicted captions (from beam search) displayed alongside the OCR text extracted from each image and the first reference caption. Showing the OCR helps interpret model behaviour — when the model produces a relevant word that appears in the OCR but not in the visual content alone, that's evidence the OCR pathway is contributing meaningfully.


In [ ]:
# Generate & Display Sample Captions (Test Set, Beam Search)
def encode_ocr_single(text, vocab, max_len=OCR_MAX_LEN):
    tokens = text.split()[:max_len]
    ids = [vocab.word2idx.get(t, vocab.unk_idx) for t in tokens]
    if not ids: ids = [vocab.pad_idx]
    return torch.tensor(ids, dtype=torch.long), torch.tensor([len(ids)], dtype=torch.long)

def caption_image_beam(model, img_tensor, ocr_text, vocab, beam_size=BEAM_SIZE):
    model.eval()
    with torch.no_grad():
        img_feats = model.encoder(img_tensor.unsqueeze(0).to(device))
        ocr_ids, ocr_lens = encode_ocr_single(ocr_text, vocab)
        ocr_feats = model.ocr_encoder(ocr_ids.unsqueeze(0), ocr_lens)
        ids = model.decoder.generate_beam(img_feats, ocr_feats, vocab, beam_size=beam_size)[0]
    return vocab.denumericalize(ids)

sample_records = random.sample(by_split['test'], 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.ravel(), sample_records):
    img = Image.open(IMG_DIR / r['filename']).convert('RGB')
    ocr_text = ocr_dict.get(r['filename'], '')
    pred = caption_image_beam(model, eval_tf(img), ocr_text, vocab)
    ax.imshow(img); ax.axis('off')
    ref = r['cleaned_captions'][0][:50]
    ocr_short = (ocr_text[:40] + '...') if len(ocr_text) > 40 else (ocr_text or '(no OCR)')
    ax.set_title(f'PRED: {pred[:50]}\nREF : {ref}\nOCR : {ocr_short}', fontsize=8, loc='left')
plt.tight_layout(); plt.show()

---
# **7. Model 1 — BLEU Evaluation**
---


**Methodology**

BLEU-1 through BLEU-4 are computed on **all three splits** (train sample, full val, full test) with beam search decoding (k=5, length penalty α=0.7), so we can diagnose overfitting/underfitting at the BLEU level — not just at the loss level.

- **Train BLEU** on a 500-image random sample (full train would take ~3-4× as long with beam search, for marginal extra insight).
- **Val BLEU** on all 1,098 val images.
- **Test BLEU** on all 733 test images — final reported metric.

**Interpretation guide**

| Pattern | Diagnosis |
|---|---|
| Train ≫ Val ≈ Test | **Overfitting** — the OCR fusion may have given the decoder too much capacity for this dataset size |
| Train ≈ Val ≈ Test (all moderate-high) | **Healthy fit** |
| Train ≈ Val ≈ Test (all low) | **Underfitting** — beam search isn't recovering quality; model needs more capacity / epochs |
| Val ≫ Test or Test ≫ Val | Splits aren't representative of each other |

**Why test is the *headline* metric**: train and val are used (sampled or fully) during training-time monitoring; reporting them as headline numbers would inflate the result. The test split is the only one untouched during training, so its BLEU is the correct final number for the report.


In [ ]:
# Compute Corpus + Per-Image BLEU on Train (sample) + Val + Test (Beam Search)
def evaluate_bleu(model, records, vocab, transform, ocr_dict,
                  max_eval=None, beam_size=BEAM_SIZE, split_name='test'):
    """Generate captions (beam search) for each record and compute corpus + per-image BLEU."""
    model.eval()
    sample = records[:max_eval] if max_eval else records
    hypotheses, references, per_image_rows = [], [], []
    smoothie = SmoothingFunction().method1

    for r in tqdm(sample, desc=f'gen ({split_name})'):
        img = transform(Image.open(IMG_DIR / r['filename']).convert('RGB')).to(device)
        ocr_text = ocr_dict.get(r['filename'], '')
        with torch.no_grad():
            img_feats = model.encoder(img.unsqueeze(0))
            ocr_ids, ocr_lens = encode_ocr_single(ocr_text, vocab)
            ocr_feats = model.ocr_encoder(ocr_ids.unsqueeze(0), ocr_lens)
            ids = model.decoder.generate_beam(img_feats, ocr_feats, vocab, beam_size=beam_size)[0]
        hyp_tokens = vocab.denumericalize(ids).split()
        ref_tokens = [tokenize(c) for c in r['cleaned_captions']]
        hypotheses.append(hyp_tokens)
        references.append(ref_tokens)
        per_image_rows.append({
            'image_id'   : r['image_id'],
            'filename'   : r['filename'],
            'hypothesis' : ' '.join(hyp_tokens),
            'ocr_text'   : ocr_text[:80],
            'hyp_len'    : len(hyp_tokens),
            'ref_len'    : float(np.mean([len(rt) for rt in ref_tokens])),
            'bleu1': sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0),       smoothing_function=smoothie),
            'bleu2': sentence_bleu(ref_tokens, hyp_tokens, weights=(0.5, 0.5, 0, 0),   smoothing_function=smoothie),
            'bleu3': sentence_bleu(ref_tokens, hyp_tokens, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoothie),
            'bleu4': sentence_bleu(ref_tokens, hyp_tokens, weights=(0.25,)*4,          smoothing_function=smoothie),
            'reference_1': r['cleaned_captions'][0],
        })

    weights = [(1.0, 0, 0, 0), (0.5, 0.5, 0, 0), (1/3, 1/3, 1/3, 0), (0.25,)*4]
    scores = {}
    print(f'\nCorpus BLEU scores on {split_name}:')
    for n, w in enumerate(weights, 1):
        s = corpus_bleu(references, hypotheses, weights=w, smoothing_function=smoothie)
        scores[f'BLEU-{n}'] = s
        print(f'  BLEU-{n}: {s:.4f}')
    return scores, hypotheses, references, pd.DataFrame(per_image_rows)


# Train BLEU (sampled)
TRAIN_SAMPLE_SIZE = 500
train_sample = random.Random(SEED).sample(by_split['train'], min(TRAIN_SAMPLE_SIZE, len(by_split['train'])))
print(f'--- Evaluating BLEU on TRAIN ({len(train_sample)} random sample) ---')
train_scores, _, _, _ = evaluate_bleu(model, train_sample, vocab, eval_tf, ocr_dict, split_name='train')

# Val BLEU (full)
print(f'\n--- Evaluating BLEU on VAL ({len(by_split["val"])} images) ---')
val_scores, _, _, _ = evaluate_bleu(model, by_split['val'], vocab, eval_tf, ocr_dict, split_name='val')

# Test BLEU (full) — headline
print(f'\n--- Evaluating BLEU on TEST ({len(by_split["test"])} images) — headline metric ---')
scores, hyps, refs, per_image_df = evaluate_bleu(model, by_split['test'], vocab, eval_tf, ocr_dict, split_name='test')

all_scores = {'train (sample)': train_scores, 'val': val_scores, 'test': scores}
print('\n=== Three-way BLEU comparison ===')
comparison_df = pd.DataFrame({
    name: [all_scores[name][f'BLEU-{n}'] for n in range(1, 5)]
    for name in all_scores
}, index=[f'BLEU-{n}' for n in range(1, 5)])
print(comparison_df.round(4).to_string())

### **7.1 Train vs Val vs Test — Overfit/Underfit Diagnostic**

**Methodology**: grouped bar chart showing BLEU-1 through BLEU-4 for all three splits side-by-side, with a printed gap analysis quantifying each split-to-split difference.


In [ ]:
# Grouped Bar Chart — Train vs Val vs Test BLEU (Beam Search)
metrics = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']
splits = list(all_scores.keys())
colors = {'train (sample)': '#2E86AB', 'val': '#F4A261', 'test': '#E76F51'}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.27

for i, split in enumerate(splits):
    vals = [all_scores[split][m] for m in metrics]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=split, color=colors[split], edgecolor='white', linewidth=1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('Corpus BLEU')
ax.set_title(f'Model 1 — BLEU on Train (sample) vs Val vs Test (beam={BEAM_SIZE})', fontweight='bold')
ax.legend(title='Split', loc='upper right')
ax.set_ylim(0, max(max(all_scores[s].values()) for s in splits) * 1.20)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

print('\nGap analysis:')
for m in metrics:
    print(f'  {m:8s} train={train_scores[m]:.4f}  val={val_scores[m]:.4f}  test={scores[m]:.4f}')
print()
print(f'  train-val BLEU-4 gap: {train_scores["BLEU-4"]-val_scores["BLEU-4"]:+.4f}')
print(f'  val-test BLEU-4 gap : {val_scores["BLEU-4"]-scores["BLEU-4"]:+.4f}')

print('\nDiagnostic:')
gap_train_val = train_scores['BLEU-4'] - val_scores['BLEU-4']
gap_val_test  = val_scores['BLEU-4'] - scores['BLEU-4']
if gap_train_val > 0.05:
    print(f'  Overfitting: train-val BLEU-4 gap = {gap_train_val:+.4f}')
elif gap_train_val > 0.02:
    print(f'  Mild overfitting: train-val BLEU-4 gap = {gap_train_val:+.4f}')
else:
    print(f'  Healthy fit: train-val BLEU-4 gap = {gap_train_val:+.4f}')
if abs(gap_val_test) > 0.03:
    print(f'  Val/test diverge (gap = {gap_val_test:+.4f}) -- flag in limitations.')
else:
    print(f'  Val and test agree (gap = {gap_val_test:+.4f}) -- splits are representative.')

### **7.2 Per-Image BLEU Distribution**

**Methodology**: corpus BLEU is a single average. Four histograms reveal whether the model is uniformly mediocre or bimodally good-or-bad on the test set.


In [ ]:
# Per-Image BLEU Distribution (Test)
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, n in zip(axes.ravel(), [1, 2, 3, 4]):
    col = f'bleu{n}'
    ax.hist(per_image_df[col], bins=30, color='steelblue', edgecolor='white')
    m, md = per_image_df[col].mean(), per_image_df[col].median()
    ax.axvline(m,  ls='--', color='red',    label=f'mean={m:.3f}')
    ax.axvline(md, ls='--', color='orange', label=f'median={md:.3f}')
    ax.set_xlabel(f'BLEU-{n}'); ax.set_ylabel('# images')
    ax.set_title(f'Per-image BLEU-{n}'); ax.legend(loc='upper right', fontsize=9)
fig.suptitle('Model 1 — Per-image BLEU distributions (test set, beam search)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nPer-image BLEU statistics (test set):')
print(per_image_df[['bleu1', 'bleu2', 'bleu3', 'bleu4']].describe().round(4))

### **7.3 Best vs Worst Predictions**

**Methodology**: the four highest-BLEU-1 and four lowest-BLEU-1 predictions on the test set, shown with image, OCR text, prediction, and first reference. Useful for distinguishing OCR-driven successes (where OCR text clearly helped) from visual-only successes or failures.


In [ ]:
# Best vs Worst Predictions (Test)
TOP_N = 4
best  = per_image_df.nlargest(TOP_N,  'bleu1').reset_index(drop=True)
worst = per_image_df.nsmallest(TOP_N, 'bleu1').reset_index(drop=True)

def show_predictions(df, title):
    fig, axes = plt.subplots(1, TOP_N, figsize=(4*TOP_N, 5))
    if TOP_N == 1: axes = [axes]
    for ax, (_, row) in zip(axes, df.iterrows()):
        img = Image.open(IMG_DIR / row['filename'])
        ax.imshow(img); ax.axis('off')
        pred = row['hypothesis'][:50] + ('...' if len(row['hypothesis']) > 50 else '')
        ref  = row['reference_1'][:50] + ('...' if len(row['reference_1']) > 50 else '')
        ocr  = row['ocr_text'][:50] + ('...' if len(row['ocr_text']) > 50 else (row['ocr_text'] or '(no OCR)'))
        ax.set_title(f'BLEU-1: {row["bleu1"]:.3f}\nPRED: {pred}\nREF : {ref}\nOCR : {ocr}',
                     fontsize=8, loc='left', pad=8)
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

show_predictions(best,  f'TOP {TOP_N} BEST predictions (highest BLEU-1)')
show_predictions(worst, f'TOP {TOP_N} WORST predictions (lowest BLEU-1)')

### **7.4 Caption Length Comparison**

**Methodology**: predicted vs reference caption lengths (overlaid histogram + boxplot). Beam search with a length penalty should produce captions closer to the reference length than greedy decoding typically does.


In [ ]:
# Caption Length Comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(per_image_df['hyp_len'], bins=20, alpha=0.6, label='Predicted (beam search)', color='steelblue', edgecolor='white')
axes[0].hist(per_image_df['ref_len'], bins=20, alpha=0.6, label='Reference (mean of 5)', color='coral', edgecolor='white')
axes[0].axvline(per_image_df['hyp_len'].mean(), ls='--', color='steelblue', label=f'pred mean={per_image_df["hyp_len"].mean():.1f}')
axes[0].axvline(per_image_df['ref_len'].mean(), ls='--', color='coral',     label=f'ref mean={per_image_df["ref_len"].mean():.1f}')
axes[0].set_xlabel('caption length (tokens)'); axes[0].set_ylabel('# images')
axes[0].set_title('Length distribution'); axes[0].legend(fontsize=9)
axes[1].boxplot([per_image_df['hyp_len'], per_image_df['ref_len']],
                tick_labels=['Predicted', 'Reference'], patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue'))
axes[1].set_ylabel('caption length (tokens)')
axes[1].set_title('Length: model vs reference'); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Predicted lengths : mean={per_image_df["hyp_len"].mean():.1f}  median={per_image_df["hyp_len"].median():.1f}  range=[{per_image_df["hyp_len"].min()}, {per_image_df["hyp_len"].max()}]')
print(f'Reference lengths : mean={per_image_df["ref_len"].mean():.1f}  median={per_image_df["ref_len"].median():.1f}')

### **7.5 Vocabulary Usage**

**Methodology**: top-20 words used by the model vs by the references, plus a diversity ratio. OCR-augmented models often have higher diversity than pure-visual models on text-heavy datasets because they can reproduce text-specific tokens from the OCR signal.


In [ ]:
# Vocabulary Usage
hyp_words = Counter(t for h in hyps for t in h)
ref_words = Counter(t for refs_ in refs for r in refs_ for t in r)

print(f'Unique words used by MODEL      : {len(hyp_words):,}')
print(f'Unique words in REFERENCES      : {len(ref_words):,}')
print(f'Diversity ratio (model / ref)   : {len(hyp_words)/max(len(ref_words),1):.2%}')

TOP_N = 20
top_pred = hyp_words.most_common(TOP_N)
top_ref  = ref_words.most_common(TOP_N)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
words, counts = zip(*top_pred)
axes[0].barh(list(reversed(words)), list(reversed(counts)), color='steelblue', edgecolor='white')
axes[0].set_xlabel('Count'); axes[0].set_title(f'Top {TOP_N} words — PREDICTED (Model 1, beam search)')
words, counts = zip(*top_ref)
axes[1].barh(list(reversed(words)), list(reversed(counts)), color='coral', edgecolor='white')
axes[1].set_xlabel('Count'); axes[1].set_title(f'Top {TOP_N} words — REFERENCE captions')
plt.tight_layout(); plt.show()

### **7.6 Summary Table**

**Methodology**: all metrics and training metadata gathered in one dataframe — drops straight into the Results section of the report.


In [ ]:
# Summary Table (Model 1)
summary_m1 = pd.DataFrame([
    ('Architecture',              'ResNet-101 (frozen) + OCR-LSTM + LSTM decoder + Beam search'),
    ('Headline evaluation split', 'test (held-out, never seen during training)'),
    # Headline (test)
    ('Corpus BLEU-1 (test)',      f'{scores["BLEU-1"]:.4f}'),
    ('Corpus BLEU-2 (test)',      f'{scores["BLEU-2"]:.4f}'),
    ('Corpus BLEU-3 (test)',      f'{scores["BLEU-3"]:.4f}'),
    ('Corpus BLEU-4 (test)',      f'{scores["BLEU-4"]:.4f}'),
    # Comparison BLEU
    ('Corpus BLEU-1 (val)',          f'{val_scores["BLEU-1"]:.4f}'),
    ('Corpus BLEU-4 (val)',          f'{val_scores["BLEU-4"]:.4f}'),
    ('Corpus BLEU-1 (train sample)', f'{train_scores["BLEU-1"]:.4f}'),
    ('Corpus BLEU-4 (train sample)', f'{train_scores["BLEU-4"]:.4f}'),
    # Per-image (test)
    ('Mean per-image BLEU-1 (test)',   f'{per_image_df["bleu1"].mean():.4f}'),
    ('Mean per-image BLEU-4 (test)',   f'{per_image_df["bleu4"].mean():.4f}'),
    ('Median per-image BLEU-1 (test)', f'{per_image_df["bleu1"].median():.4f}'),
    # Length / vocab
    ('Mean predicted length',     f'{per_image_df["hyp_len"].mean():.1f} tokens'),
    ('Mean reference length',     f'{per_image_df["ref_len"].mean():.1f} tokens'),
    ('Unique words (predicted)',  f'{len(hyp_words):,}'),
    ('Unique words (reference)',  f'{len(ref_words):,}'),
    ('Vocabulary diversity',      f'{len(hyp_words)/max(len(ref_words),1):.2%}'),
    # OCR-specific
    ('OCR coverage',              f'{sum(1 for v in ocr_dict.values() if v):,}/{len(ocr_dict):,} images ({sum(1 for v in ocr_dict.values() if v)/len(ocr_dict)*100:.1f}%)'),
    ('Beam size',                 f'{BEAM_SIZE}'),
    ('Length penalty α',          f'{LENGTH_PENALTY_ALPHA}'),
    # Eval sample sizes
    ('# train images evaluated',  f'{len(train_sample)} (sampled from 5,493)'),
    ('# val images evaluated',    f'{len(by_split["val"]):,}'),
    ('# test images evaluated',   f'{len(per_image_df):,}'),
    # Training
    ('# training epochs',         f'{EPOCHS}'),
    ('Learning rate',             f'{LR}'),
    ('Trainable params',          f'{n_trainable:,}  (OCR encoder + decoder)'),
    ('Total parameters',          f'{n_total:,}'),
    ('Device',                    str(device)),
], columns=['Metric', 'Value'])
print(summary_m1.to_string(index=False))

---
# **8. Model 1 — Summary & Limitations**
---


**What Model 1 achieved** *(fill in after the training run completes)*

- Corpus BLEU-1 (test): ___ / BLEU-2: ___ / BLEU-3: ___ / BLEU-4: ___
- Per-image median BLEU-1 (test): ___
- Train vs val vs test BLEU-4: ___ vs ___ vs ___ → **Diagnosis: overfit / healthy / underfit**
- OCR coverage: ___% of images had non-empty OCR text
- Trainable parameters: ___ (OCR encoder + decoder; image encoder frozen)
- Total training + OCR extraction time: ___ minutes on ___ device.

**Observations** *(fill in after running)*

- Did beam search produce noticeably less generic captions than greedy would? ___
- Train-val BLEU gap: ___ (large = overfitting on the small training set)
- Did predictions on text-heavy images (medication, packaging) include OCR-derived words? ___
- Worst predictions tend to be on: ___ (images without OCR? specific scene types?)
- Best predictions tend to be on: ___ (was OCR clearly contributing?)

**Limitations of Model 1**

| Limitation | Impact | Possible Phase 3 fix |
|---|---|---|
| Image encoder fully frozen | Cannot adapt visual features to VizWiz-specific content | Two-stage warmup → fine-tune schedule (matches student 3 approach) |
| OCR vocab reuses caption vocab | OCR-specific tokens (brand names, technical terms) become `<unk>` | Build a separate OCR vocabulary or use sub-word tokenisation |
| OCR processed paragraph-mode | May lose layout information (e.g. heading vs body text) | Use detail mode with bounding boxes; encode positionally |
| Single-layer LSTM decoder | Limited long-range capacity | 2-layer LSTM with inter-layer dropout |
| OCR encoded once; not attended | Decoder cannot focus on specific OCR words per timestep | Add attention from decoder to OCR token sequence (like image attention in Show-Attend-Tell) |
| EasyOCR English-only | Foreign-language text in VizWiz becomes empty OCR | Multi-language EasyOCR or fallback to another OCR engine |
| Greedy-vs-beam not directly compared | Cannot quantify beam contribution | Add a greedy evaluation run for direct comparison |

**Hypotheses to test in the group discussion**

1. **Did OCR help?** Compare against teammate #1 (LSTM + fine-tune, no OCR) — does my Model 1 (LSTM + OCR, no fine-tune) score higher on test BLEU?
2. **OCR vs Transformer**: did my OCR-LSTM beat teammate #3's Transformer-with-fine-tuning?
3. **Did beam search make a measurable difference**? Teammates #2, #3, #5 used greedy — is there a systematic BLEU gap?
4. **Where does OCR most help?** Look at images where my BLEU is high and OCR text is non-empty — are these the text-heavy images (packaging, screens, mail)?


---
# **9. Group Discussion Notes**
---


**Meeting date**: TBD

**Attending**: [your name] + [other group members]

**Each member's Model 1 results** *(fill in)*

| Student | Architecture | BLEU-1 (test) | BLEU-2 | BLEU-3 | BLEU-4 | Notes |
|---|---|---|---|---|---|---|
| 1 | ResNet-101 + 2-LSTM + fine-tune + beam | | | | | |
| 2 | ResNet-101 + Attn + 2-LSTM + fine-tune + greedy | | | | | |
| 3 | ResNet-101 + Transformer + Warmup→Fine-tune + greedy | | | | | |
| **4 (me)** | **ResNet-101 + OCR + LSTM + beam** | | | | | |
| 5 | ResNet-101 + OCR + Transformer + greedy | | | | | |

**Comparison axes for analysis**

- #1 vs #4 (me) — does OCR augmentation beat encoder fine-tuning?
- #4 (me) vs #5 — within OCR architectures, does LSTM (with beam) or Transformer (with greedy) win?
- #1 vs #4 (me) — both use beam search, isolating OCR's contribution
- #2 vs #4 (me) — both LSTM-based, but attention (theirs) vs OCR fusion (mine)

**Key findings from the comparison** *(fill in)*

1. Best-performing model: ___
2. Most impactful design choice: ___
3. Did OCR consistently help? ___
4. Did beam search consistently help? ___
5. Common failure modes across all models: ___

**Implications for my Model 2**

Based on the discussion, the changes I will make for Model 2 are:

- ___
- ___
- ___


---
# **10. Model 2 — Refined Architecture (TBD)**
---


**Status**: Pending group discussion of all 5 students' Model 1 results.

**Candidate Model 2 directions** (final choice depends on group discussion findings)

| Direction | Why it might help | Cost |
|---|---|---|
| **Add OCR attention** (decoder attends to OCR tokens at each step) | Currently OCR is fused once at the LSTM init; per-step attention lets the decoder focus on relevant OCR words while generating | Moderate (adds an attention module similar to Show-Attend-Tell) |
| **Encoder fine-tuning** (last ResNet block) | Adapts visual features to VizWiz characteristics; would let me compare OCR-only vs OCR-plus-fine-tune | Moderate (two-stage training schedule like student 3) |
| **2-layer LSTM decoder with inter-layer dropout** | Increases language-modelling capacity for longer/more complex captions | Trivial code change |
| **Multi-language OCR** (EasyOCR with broader language pack) | Captures non-English text in VizWiz images | Easy — change EasyOCR config; re-extract OCR cache |
| **Pre-trained word embeddings** (GloVe / FastText) for both decoder + OCR encoder | Better generalisation to OOV-adjacent OCR words | Moderate (need to align with vocab) |
| **Larger beam (k=10) with diverse beam search** | More candidates explored; diverse beam search reduces near-duplicate beams | Easy code change; longer inference |

**Sections to add when Model 2 is built** (mirroring Model 1 layout):

- 11. Model 2 — Architecture
- 12. Model 2 — Training
- 13. Model 2 — Sample Generations (test set)
- 14. Model 2 — BLEU Evaluation (train/val/test comparison, same structure as section 7)
- 15. Model 2 — Summary & Comparison vs Model 1
- 16. Final Conclusions for the Report
